In [105]:
import os

os.makedirs("output", exist_ok=True)

# Part 1 - Read and clean

In [106]:
import pandas as pd
import numpy as np

In [107]:
# 1
sheet1 = pd.read_excel("data/eurostat.xlsx", sheet_name="Sheet 1", skiprows=9, usecols=[0] + list(range(1,46,2)), na_values=":", nrows=44)
print(sheet1.shape)
print(sheet1.dtypes)

(44, 24)
TIME        str
2001    float64
2002    float64
2003    float64
2004    float64
2005    float64
2006    float64
2007    float64
2008    float64
2009    float64
2010    float64
2011    float64
2012    float64
2013    float64
2014    float64
2015    float64
2016    float64
2017    float64
2018    float64
2019    float64
2020    float64
2021    float64
2022    float64
2023    float64
dtype: object


In [108]:
# 2
sheet1 = sheet1.rename(columns={"TIME": "country"})
sheet1 = sheet1[sheet1["country"] != "GEO (Labels)"].reset_index(drop=True) # removes geo row

print("rows before:", pd.read_excel("data/eurostat.xlsx", sheet_name="Sheet 1", skiprows=9).shape[0])
print("rows after:", sheet1.shape[0])


rows before: 49
rows after: 43


In [109]:
# 3

# all aggregates contains the word "contries", so we use that as the mask
mask = sheet1["country"].str.contains("countries") 

aggregates = sheet1[mask].reset_index(drop=True)
countries = sheet1[~mask].reset_index(drop=True) # all countries not in mask

print(aggregates.shape)
print(countries.shape)

(2, 24)
(41, 24)


In [110]:
# 4
countries.isna().sum()

country     0
2001        5
2002        5
2003        5
2004        5
2005        4
2006        4
2007        4
2008        4
2009        4
2010        3
2011        3
2012        3
2013        2
2014        0
2015        0
2016        0
2017        0
2018        0
2019        0
2020        1
2021        1
2022        3
2023       38
dtype: int64

As we can see, 2022 is the last year where we are missing almost none of the values. In 2023 there are almost only missing data. Therefore we set 2022 as the end year for the rest of the assignment:

In [111]:
end_year = 2022

In [112]:
year_cols = []
for col in countries.columns:
    if col == "country":
        continue # skip the first column
    if int(col) <= end_year:
        year_cols.append(col)

has_gap = countries[year_cols].isna().any(axis=1)
with_gaps = countries[has_gap] # all the countries with gaps

for index, row in with_gaps.iterrows():
    missing_years = []
    for col in year_cols:
        if pd.isna(row[col]):
            missing_years.append(col)
    print(row["country"], ":", missing_years)


Iceland : ['2022']
Liechtenstein : ['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013']
United Kingdom : ['2020', '2021', '2022']
Bosnia and Herzegovina : ['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013']
Montenegro : ['2001', '2002', '2003', '2004']
Moldova : ['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009']
Georgia : ['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012']
Ukraine : ['2022']


It looks like there are some countries that are missing values in the start, and some are missing values at the end. The missing values are also continuous over several years.

# Part 2 - A function, then the pandas way

In [113]:
# 1
def percent_change(old, new):
    """
    Returns the change from old to new in percent of old.
    
    Params:
    old: float
        The starting value. Must be positive
    new: float
        The end value
        
    Returns:
    float
        the percentage change or np.nan if either value is NaN
        
    Raises:
    ValueError
        if old is zero or negative.
    """
    
    if pd.isna(old) or pd.isna(new):
        return np.nan
    if old <= 0:
        raise ValueError("old must be positive, got " + str(old))
    else:
        return 100*(new/old - 1)

In [114]:
print(percent_change(100,150))
print(percent_change(200,150))
print(percent_change(80,80))
print(percent_change(100,np.nan))
print(percent_change(np.nan,100))
try:
    percent_change(0,50)
except ValueError as e:
    print(e)
try:
    percent_change(-10,50)
except ValueError as e:
    print(e)
    


50.0
-25.0
0.0
nan
nan
old must be positive, got 0
old must be positive, got -10


In [115]:
# 2
changes = [] # a list containing all the changes from year 2001 to 2022.

for old, new in zip(countries[year_cols[0]], countries[year_cols[-1]]):
    changes.append(percent_change(old, new))

countries["pct_change"] = changes # creates a new col with values from changes list

In [116]:
# 3
old = countries[year_cols[0]] # 2001 column
new = countries[year_cols[-1]] # 2022 column

countries["pct_change2"] = (new/old - 1) * 100

In [117]:
# Are they missing in the same rows?
same_missing = (countries["pct_change"].isna() == countries["pct_change2"].isna()).all()
print("same missing: ",same_missing)

# What is the largest difference where both have a value?
difference = (countries["pct_change"] - countries["pct_change2"]).abs().max()
print("largest difference: ", difference)

# Do the two columns agree?
agree = same_missing and difference.round(6) == 0
print("agree: ", agree)

same missing:  True
largest difference:  0.0
agree:  True


As we can see, the two columns (pct_change and pct_change2) are identical. I would choose the vectorized version (version 2) because it is shorter and faster. This is because pandas does the calculation on the entire column at once instead of running the code for every country.

The function is still useful for single pairs of numbers, and because it checks its input. The vectorized version does not raise ValueErrors like the function does.

In [118]:
countries = countries.drop(columns="pct_change2") # remove the extra column (not needed)

# Part 3 - Classify, rank and check

In [119]:
# 1
conditions = [
    countries["pct_change"] >= 20,
    countries["pct_change"] >= 5,
    countries["pct_change"] > -5,
    countries["pct_change"] <= -5,
]

choices = ["strong growth", "growth", "stable", "decline"]

countries["trend"] = np.select(conditions, choices, default="no data")

In [120]:
countries["trend"].value_counts()

trend
strong growth    14
growth            9
stable            8
no data           8
decline           2
Name: count, dtype: int64

In [121]:
# 2

# Sort table based on pct_change, and find the first and last 5
sorted_countries = countries.dropna(subset="pct_change").sort_values("pct_change", ascending=False)[["country", "pct_change"]]

# Top 5 largest rise
largest_rise = sorted_countries[0:5]
print("largest_rise: \n", largest_rise, "\n")

# Top 5 largest fall
largest_fall = sorted_countries[::-1][0:5] 
print("largest_fall: \n", largest_fall)

largest_rise: 
       country  pct_change
38    Türkiye  195.232536
17      Malta   71.856469
36    Albania   68.942718
14  Lithuania   57.113459
12     Cyprus   55.912903 

largest_fall: 
      country  pct_change
40   Kosovo*  -81.259276
26    Sweden   -6.575536
4    Germany   -4.546414
3    Denmark   -2.521597
24  Slovakia   -1.544465


In [122]:
# 3
largest_change = None
largest_country = None
largest_year = None

# Loop through each row. 
for i in range(len(countries)): # the rows
    for j in range(len(year_cols)-1): # the columns
        old = countries[year_cols[j]].iloc[i] # first col for row i
        new = countries[year_cols[j+1]].iloc[i] # next col for row i

        change=percent_change(old, new)
        if largest_change is None or abs(change) > abs(largest_change):
            largest_change = change
            largest_country = countries["country"][i]
            largest_year = [year_cols[j], year_cols[j+1]]

print("The largest change was", 
      largest_change.round(2), "%. \nIt was in", 
      largest_country, "from", 
      largest_year[0], "to", largest_year[1])



The largest change was -90.76 %. 
It was in Kosovo* from 2021 to 2022


The largest change was -90.76 % in kosovo. Let us see if that is plausible:


In [123]:
kosovo = countries[countries["country"] == "Kosovo*"] # filter on kosovo
kosovo[["country", "2019", "2020", "2021", "2022"]] # show only surrounding years

,country,2019,2020,2021,2022
40,Kosovo*,4728.863,4898.293,5938.266,548.541


As we can see, the values go from almost 6000 to 548 from 2021 to 2022. This is a huge fall, and it looks like there is something wrong with this data.

Let us fix it by setting 2022 to NaN and trying again:

In [124]:
countries.loc[countries["country"] == "Kosovo*", "2022"] = np.nan

Then we calculate the pct_changes again and classify the trends: 

In [125]:
# pct_change column
changes = [] # a list containing all the changes from year 2001 to 2022.

for old, new in zip(countries[year_cols[0]], countries[year_cols[-1]]):
    changes.append(percent_change(old, new))

countries["pct_change"] = changes # creates a new col with values from changes list

# classification
conditions = [
    countries["pct_change"] >= 20,
    countries["pct_change"] >= 5,
    countries["pct_change"] > -5,
    countries["pct_change"] <= -5,
]

choices = ["strong growth", "growth", "stable", "decline"]

countries["trend"] = np.select(conditions, choices, default="no data")

In [126]:
countries[countries["country"] == "Kosovo*"]

,country,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,pct_change,trend
40,Kosovo*,2927.0,2908.0,2843.0,2998.0,3239.0,3363.0,3651.0,4077.0,3922.0,...,3870.0,4346.347,4403.938,4728.863,4898.293,5938.266,NaN,NaN,NaN,no data


Lets try task 2 again (top 5 task)

In [127]:
# 3.2

# Sort table based on pct_change, and find the first and last 5
sorted_countries = countries.dropna(subset="pct_change").sort_values("pct_change", ascending=False)[["country", "pct_change"]]

# Top 5 largest rise
largest_rise = sorted_countries[0:5]
print("largest_rise: \n", largest_rise, "\n")

# Top 5 largest fall
largest_fall = sorted_countries[::-1][0:5] 
print("largest_fall: \n", largest_fall)

largest_rise: 
       country  pct_change
38    Türkiye  195.232536
17      Malta   71.856469
36    Albania   68.942718
14  Lithuania   57.113459
12     Cyprus   55.912903 

largest_fall: 
      country  pct_change
26    Sweden   -6.575536
4    Germany   -4.546414
3    Denmark   -2.521597
24  Slovakia   -1.544465
0    Belgium   -0.338529


The new top 5 lists show that the correction changed the answer to question 2. Kosovo was the country with the largest fall at about -81% before, caused by the 2022 value. Kosovo is now no longer ranked because its 2022 value is now missing, so it has no pct_change.

In [128]:
# 4
eu_members = [
    "Belgium", "Bulgaria", "Czechia", "Denmark", "Germany", "Estonia", "Ireland",
    "Greece", "Spain", "France", "Croatia", "Italy", "Cyprus", "Latvia",
    "Lithuania", "Luxembourg", "Hungary", "Malta", "Netherlands", "Austria",
    "Poland", "Portugal", "Romania", "Slovenia", "Slovakia", "Finland", "Sweden",
]

members = countries[countries["country"].isin(eu_members)]
eu_countries = aggregates.iloc[0]
largest_diff = None
diff_year = None

for year in year_cols:
    aggregate_sum = eu_countries[year]
    members_sum = sum(members[year])

    diff = abs(aggregate_sum-members_sum)

    if largest_diff is None or diff > largest_diff:
        largest_diff = diff
        diff_year = year

print("members found:", len(members))
print("missing values among members:", members[year_cols].isna().sum().sum())

print("Largest difference: ", largest_diff, "in year", diff_year)

members found: 27
missing values among members: 0
Largest difference:  4.656612873077393e-10 in year 2010


The largest difference is approximately zero, so the two agree. 

To make sure the comparison was fair, I checked that the amount of members was 27 and that no values were missing.

In [129]:
# 5 Exporting the countries table
countries.to_csv("output/electricity_clean.csv", index=False)

In [130]:
# Check that it worked:
test = pd.read_csv("output/electricity_clean.csv")
print(test.shape, countries.shape)
print(list(test.columns) == list(countries.columns))

test.head()

(41, 26) (41, 26)
True


,country,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,pct_change,trend
0,Belgium,79816.0,80438.0,82065.0,83034.5,82983.1,85663.1,85675.50,85885.900,80007.300,...,83183.700,83766.200,84222.800,83291.300,80897.500,83912.400,79545.800,74812.000,-0.338529,stable
1,Bulgaria,25776.0,25287.0,26425.0,26241.0,27213.0,28444.0,29286.00,29923.000,28259.000,...,30259.000,31445.670,31389.999,31094.533,30498.913,31700.223,31205.105,30874.342,21.062636,strong growth
2,Czechia,53775.0,53669.0,54807.0,56388.0,57664.0,59422.0,59740.01,60463.654,57098.304,...,59762.129,60931.039,61327.781,61193.576,59111.206,62158.363,59937.858,NaN,11.460452,growth
3,Denmark,33213.0,33173.0,33159.0,33719.0,34254.0,34681.0,34458.00,34132.000,32425.539,...,32061.184,32255.674,31896.936,32192.383,31479.924,33740.355,32375.502,NaN,-2.521597,stable
4,Germany,512637.0,525510.0,531751.0,538406.0,539471.0,544629.0,545614.00,542991.000,513898.000,...,530551.000,531318.000,521809.000,507543.000,492821.000,510100.500,489330.400,NaN,-4.546414,stable


It looks good based on the shape and that the columns are the same, and that the data looks correct.

# Part 4 - Merge and summarize

In [131]:
# 1
co2 = pd.read_csv("data/co2_emissions.csv")

co2 = co2[co2["year"]==end_year][["country", "code", "population"]]
merged = countries.merge(co2, on="country", how="left", validate="one_to_one", indicator=True)

merged["_merge"].value_counts()

_merge
both          38
left_only      3
right_only     0
Name: count, dtype: int64

As we can see, 38 worked, 3 did not work. Let's see why:

In [132]:
unmatched = merged[merged["_merge"] == "left_only"]["country"]
print(list(unmatched))


['Slovakia', 'Türkiye', 'Kosovo*']


The problem with these names is that they may have a different name in the co2 file. Lets investigate:

In [133]:
co2_all = pd.read_csv("data/co2_emissions.csv")

letters = ["Slov", "Tu", "Kos"]

for c in letters:
    candidates = co2_all[co2_all["country"].str.contains(c)]["country"].unique()
    print(c, "gives", list(candidates))

Slov gives ['Slovak Republic', 'Slovenia']
Tu gives ['Tunisia', 'Turkiye', 'Turkmenistan', 'Turks and Caicos Islands', 'Tuvalu']
Kos gives ['Kosovo']


So:
    Slovakia = Slovak Republic, Türkiye = Turkiye, Kosovo* = Kosovo

The problem was therefore that the names didnt match. Let us rename these countries in the co2 file:

In [134]:
co2.loc[co2["country"] == "Kosovo", "country"] = "Kosovo*"
co2.loc[co2["country"] == "Turkiye", "country"] = "Türkiye"
co2.loc[co2["country"] == "Slovak Republic", "country"] = "Slovakia"


In [135]:
merged = countries.merge(co2, on="country", how="left", validate="one_to_one", indicator=True)

merged["_merge"].value_counts()

_merge
both          41
left_only      0
right_only     0
Name: count, dtype: int64

It worked. 

In [136]:
country_info = pd.read_csv("data/country_info.csv")
country_info.head()

,name,code,region,incomeLevel
0,Afghanistan,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income
1,Africa,AFR,Aggregates,Aggregates
2,Africa Eastern and Southern,AFE,Aggregates,Aggregates
3,Africa Western and Central,AFW,Aggregates,Aggregates
4,Albania,ALB,Europe & Central Asia,Upper middle income


I first tried merging on the name column (I didnt have the code column at first). However that was difficult because there was different country name spellings on three of the countries. Therefore I tried with the "code" column instead, and that worked:

In [137]:

merged = merged.merge(
    country_info[["code", "incomeLevel"]], 
    on="code",
    how="left",
    validate="one_to_one"
    )

print(merged["incomeLevel"].isna().sum())
merged["incomeLevel"].value_counts()


0


incomeLevel
High income            31
Upper middle income    10
Name: count, dtype: int64

In [138]:
# 3
merged["eu_member"] = merged["country"].isin(eu_members)

merged.head()

,country,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2021,2022,2023,pct_change,trend,code,population,_merge,incomeLevel,eu_member
0,Belgium,79816.0,80438.0,82065.0,83034.5,82983.1,85663.1,85675.50,85885.900,80007.300,...,83912.400,79545.800,74812.000,-0.338529,stable,BEL,11680210,both,High income,True
1,Bulgaria,25776.0,25287.0,26425.0,26241.0,27213.0,28444.0,29286.00,29923.000,28259.000,...,31700.223,31205.105,30874.342,21.062636,strong growth,BGR,6465097,both,High income,True
2,Czechia,53775.0,53669.0,54807.0,56388.0,57664.0,59422.0,59740.01,60463.654,57098.304,...,62158.363,59937.858,NaN,11.460452,growth,CZE,10672118,both,High income,True
3,Denmark,33213.0,33173.0,33159.0,33719.0,34254.0,34681.0,34458.00,34132.000,32425.539,...,33740.355,32375.502,NaN,-2.521597,stable,DNK,5903037,both,High income,True
4,Germany,512637.0,525510.0,531751.0,538406.0,539471.0,544629.0,545614.00,542991.000,513898.000,...,510100.500,489330.400,NaN,-4.546414,stable,DEU,83177813,both,High income,True


In [139]:
merged["kwh_pc"] = merged["2022"] * 1000000 / merged["population"]

In [140]:
sorted_by_kwhpc = merged.dropna(subset="kwh_pc").sort_values("kwh_pc", ascending=False)[["country", "kwh_pc"]]

# top 5
print("Top 5:\n", sorted_by_kwhpc.head(5), "\n")

# bottom 5
print("Bottom 5:\n", sorted_by_kwhpc[::-1][0:5])

Top 5:
           country        kwh_pc
29         Norway  22817.015620
25        Finland  14077.845167
26         Sweden  11991.485410
28  Liechtenstein  10363.861950
15     Luxembourg   9432.209008 

Bottom 5:
             country       kwh_pc
33          Moldova  1557.745741
22          Romania  2428.982290
36          Albania  2829.452659
34  North Macedonia  3270.667004
38          Türkiye  3372.283024


Now we check what countries are missing from the ranking

In [141]:
missing = merged[merged["kwh_pc"].isna()]["country"]
print(list(missing))

['Iceland', 'United Kingdom', 'Ukraine', 'Kosovo*']


These 4 are missing. Let's esimate where one of them, for example Iceland, would rank:

In [142]:
iceland = merged[merged["country"] == "Iceland"]

gwh = iceland["2021"].iloc[0]
pop = iceland["population"].iloc[0]

estimate = gwh*1000000 / pop
print("kwh_pc iceland: ", estimate)


kwh_pc iceland:  48440.873500993446


As we can see, iceland has much higher kwh_pc than the rest. This changes the picture since it should be at the top of the ranking.

In [143]:
# 4
merged["pop_used"] = merged["population"]
merged.loc[merged["2022"].isna(), "pop_used"] = np.nan

summary = merged.groupby(["eu_member", "incomeLevel"]).agg(
    countries=("country", "size"), # number of countries in the group
    with_figure=("2022", "count"), # how many have a figure
    total_gwh=("2022", "sum"), # electricity use
    total_population=("pop_used", "sum"),
    avg_kwh_pc=("kwh_pc", "mean"), 
).reset_index()

summary["kwh_pc_group"] = summary["total_gwh"] * 1000000 / summary["total_population"]

summary.to_excel("output/electricity_summary.xlsx", index=False)

summary

,eu_member,incomeLevel,countries,with_figure,total_gwh,total_population,avg_kwh_pc,kwh_pc_group
0,False,High income,4,2,124924.652,5496620.0,16590.438785,22727.540197
1,False,Upper middle income,10,8,360660.986,105997259.0,3345.476613,3402.550117
2,True,High income,27,27,2492046.419,447203596.0,5808.129371,5572.509795


The table shows electricity use per person for three groups:
- EU countries with high income
- non-EU countries with high income
- non-EU countries with upper middle income

The EU group uses about 5 600 kWh per person, the non-EU upper middle income group only about 3 400, and the non-EU high income group the most, about 22 700.

The two per-person columns differ because they weight the countries differently. The group figure divides the group's total electricity by its total population, so large countries count more. The average treats every country equally. The difference is biggest for the non-EU high income group (about 22 700 vs 16 600), because Norway has a much larger population than Liechtenstein and dominates the group figure.

I trust the non-EU high income figure the least, because only 2 of its 4 countries have a figure (Iceland and the United Kingdom are missing), so it rests on just Norway and Liechtenstein.
